# `DocLayNet` U-Net-ResNet18 Architecture
**Author**: Juan Pablo Triana Martinez.

The following notebook contains all of the `torch.nn` code to recreate a
**U-Net** architecture with a **ResNet18** encoder from scratch (~**14.3M** parameters)!

- U-Net paper: https://arxiv.org/abs/1505.04597
- ResNet paper: https://arxiv.org/abs/1512.03385

The architecture depends on 3 components:
1. A `ResNet18Encoder` backbone that extracts multi-scale features.
2. A `UNetDecoder` that upsamples 2x at every stage while **concatenating**
   the encoder skip connections (the classic U-Net "copy and crop").
3. A `3x3` segmentation head that maps the final features to `N` logit channels.


## 1. The `ResNet18` encoder backbone (from scratch)

We first rebuild the **ResNet18** feature extractor from the original paper
(https://arxiv.org/abs/1512.03385), exactly as in `src/models/backbones.py`.
It is composed of:
- A **stem**: `7x7/2` convolution followed by `3x3/2` max pooling.
- Four residual stages (`layer1..layer4`), each with two `BasicBlock`s
  (two `3x3` convolutions + identity/projection skip connection).

For an input `(B, 3, 512, 512)` the encoder returns 5 multi-scale feature maps:

```python
x  -> stem_conv          -> f1 (B,  64, 256, 256)   # 1/2
f1 -> maxpool + layer1   -> f2 (B,  64, 128, 128)   # 1/4
f2 -> layer2             -> f3 (B, 128,  64,  64)   # 1/8
f3 -> layer3             -> f4 (B, 256,  32,  32)   # 1/16
f4 -> layer4             -> f5 (B, 512,  16,  16)   # 1/32
```

We start with a shared `ConvBNReLU` helper block used across all our benchmark architectures.


In [ ]:
# Let's import all necessary modules for this architecture
from typing import List
import torch
import torch.nn as nn
import torch.nn.functional as F


In [ ]:
class ConvBNReLU(nn.Module):
    '''
    Standard Conv2d -> BatchNorm2d -> ReLU block used across all architectures.

    Args:
        m (int): number of input channels.
        n (int): number of output channels.
        kernel_size (int): convolution kernel size.
        stride (int): convolution stride.
        padding (int): convolution padding.
        dilation (int): convolution dilation.
        groups (int): convolution groups (used for depthwise convolutions).
        relu6 (bool): if True, uses ReLU6 (MobileNetV2 convention) instead of ReLU.
    '''

    def __init__(self, m: int, n: int, kernel_size: int = 3, stride: int = 1,
                 padding: int = 1, dilation: int = 1, groups: int = 1,
                 relu6: bool = False) -> None:
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_channels=m, out_channels=n, kernel_size=kernel_size,
                      stride=stride, padding=padding, dilation=dilation,
                      groups=groups, bias=False),
            nn.BatchNorm2d(num_features=n, eps=1e-05, momentum=0.1,
                           affine=True, track_running_stats=True),
            nn.ReLU6() if relu6 else nn.ReLU()
        )

    def forward(self, x) -> torch.Tensor:
        return self.block(x)


In [ ]:
class ResNetBasicBlock(nn.Module):
    '''
    Class that defines the BasicBlock of the ResNet18 architecture
    (two 3x3 convolutions with an identity or projected skip connection).
    Reference: https://arxiv.org/abs/1512.03385

    Args:
        m (int): number of input channels.
        n (int): number of output channels.
        stride (int): stride of the first convolution (2 halves the resolution).
    '''

    def __init__(self, m: int, n: int, stride: int = 1) -> None:
        super().__init__()

        # First 3x3 convolution (possibly downsampling)
        self.conv_block_1 = nn.Sequential(
            nn.Conv2d(in_channels=m, out_channels=n, kernel_size=(3, 3),
                      stride=(stride, stride), padding=(1, 1), bias=False),
            nn.BatchNorm2d(num_features=n, eps=1e-05, momentum=0.1,
                           affine=True, track_running_stats=True),
            nn.ReLU()
        )

        # Second 3x3 convolution (no activation before the residual add)
        self.conv_block_2 = nn.Sequential(
            nn.Conv2d(in_channels=n, out_channels=n, kernel_size=(3, 3),
                      stride=(1, 1), padding=(1, 1), bias=False),
            nn.BatchNorm2d(num_features=n, eps=1e-05, momentum=0.1,
                           affine=True, track_running_stats=True)
        )

        # Projection skip connection when shape changes, identity otherwise
        if stride != 1 or m != n:
            self.skip_conn = nn.Sequential(
                nn.Conv2d(in_channels=m, out_channels=n, kernel_size=(1, 1),
                          stride=(stride, stride), padding=(0, 0), bias=False),
                nn.BatchNorm2d(num_features=n, eps=1e-05, momentum=0.1,
                               affine=True, track_running_stats=True)
            )
        else:
            self.skip_conn = nn.Identity()

        self.relu = nn.ReLU()

    def forward(self, x) -> torch.Tensor:
        out = self.conv_block_1(x)
        out = self.conv_block_2(out)
        out = out + self.skip_conn(x)
        return self.relu(out)


In [ ]:
class ResNet18Encoder(nn.Module):
    '''
    Class that defines the full ResNet18 feature-extractor backbone from scratch
    (no fully connected head), returning multi-scale feature maps.
    Reference: https://arxiv.org/abs/1512.03385

    Feature maps returned for an input of shape (B, Cin, H, W):
        f1: (B,  64, H/2,  W/2)   -> after stem conv (before max pooling)
        f2: (B,  64, H/4,  W/4)   -> after layer1
        f3: (B, 128, H/8,  W/8)   -> after layer2
        f4: (B, 256, H/16, W/16)  -> after layer3
        f5: (B, 512, H/32, W/32)  -> after layer4

    Args:
        Cin (int): number of input channels (3 for RGB document images).
    '''

    # Output channels at each stage, useful for building decoders
    out_channels: List[int] = [64, 64, 128, 256, 512]

    def __init__(self, Cin: int = 3) -> None:
        super().__init__()

        # Stem: 7x7/2 convolution followed by 3x3/2 max pooling
        self.stem_conv = nn.Sequential(
            nn.Conv2d(in_channels=Cin, out_channels=64, kernel_size=(7, 7),
                      stride=(2, 2), padding=(3, 3), bias=False),
            nn.BatchNorm2d(num_features=64, eps=1e-05, momentum=0.1,
                           affine=True, track_running_stats=True),
            nn.ReLU()
        )
        self.max_pool = nn.MaxPool2d(kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))

        # Four residual stages, two BasicBlocks each (ResNet18 configuration)
        self.layer1 = nn.Sequential(
            ResNetBasicBlock(m=64, n=64, stride=1),
            ResNetBasicBlock(m=64, n=64, stride=1)
        )
        self.layer2 = nn.Sequential(
            ResNetBasicBlock(m=64, n=128, stride=2),
            ResNetBasicBlock(m=128, n=128, stride=1)
        )
        self.layer3 = nn.Sequential(
            ResNetBasicBlock(m=128, n=256, stride=2),
            ResNetBasicBlock(m=256, n=256, stride=1)
        )
        self.layer4 = nn.Sequential(
            ResNetBasicBlock(m=256, n=512, stride=2),
            ResNetBasicBlock(m=512, n=512, stride=1)
        )

    def forward(self, x) -> List[torch.Tensor]:
        f1 = self.stem_conv(x)          # (B, 64, H/2, W/2)
        f2 = self.layer1(self.max_pool(f1))  # (B, 64, H/4, W/4)
        f3 = self.layer2(f2)            # (B, 128, H/8, W/8)
        f4 = self.layer3(f3)            # (B, 256, H/16, W/16)
        f5 = self.layer4(f4)            # (B, 512, H/32, W/32)
        return [f1, f2, f3, f4, f5]


### 1.1 Summary info of `ResNet18Encoder`

In [ ]:
from torchinfo import summary
# Let's inspect the ResNet18 encoder backbone
test_model = ResNet18Encoder(Cin=3)

summary(model = test_model,
        input_size=(1, 3, 512, 512), # (batch_size, num_channels, height, width)
        col_names = ["input_size", "output_size", "num_params", "trainable"],
        col_width = 20,
        row_settings = ["var_names"],
        depth = 3
        )


## 2. The U-Net decoder

The decoder mirrors the encoder: at every stage we **bilinearly upsample 2x**,
**concatenate** the matching encoder skip feature along the channel axis, and blend with
two `3x3` conv + BatchNorm + ReLU blocks. Following the widely used channel plan
`(256, 128, 64, 32, 16)`:

```python
f5 (B, 512, 16, 16)   -> decoder_block_5(+f4, 256 skip ch) -> (B, 256,  32,  32)
                      -> decoder_block_4(+f3, 128 skip ch) -> (B, 128,  64,  64)
                      -> decoder_block_3(+f2,  64 skip ch) -> (B,  64, 128, 128)
                      -> decoder_block_2(+f1,  64 skip ch) -> (B,  32, 256, 256)
                      -> decoder_block_1 (no skip)         -> (B,  16, 512, 512)
```

**NOTE**: like the LinkNet notebook, we use `nn.Upsample`/`F.interpolate` (bilinear)
instead of `ConvTranspose2d` to avoid checkerboard artifacts.


In [ ]:
class UNetDecoderBlock(nn.Module):
    '''
    Class that defines a U-Net decoder block: bilinear 2x upsampling,
    concatenation with the encoder skip feature, and two 3x3 convolutions.

    Args:
        m (int): number of input channels (from the previous decoder stage).
        skip (int): number of channels of the encoder skip connection (0 if none).
        n (int): number of output channels.
    '''

    def __init__(self, m: int, skip: int, n: int) -> None:
        super().__init__()
        self.conv_block_1 = ConvBNReLU(m=m + skip, n=n, kernel_size=3,
                                       stride=1, padding=1)
        self.conv_block_2 = ConvBNReLU(m=n, n=n, kernel_size=3,
                                       stride=1, padding=1)

    def forward(self, x, skip=None) -> torch.Tensor:
        # Bilinear 2x upsampling (avoids checkerboard artifacts of ConvTranspose2d)
        x = F.interpolate(x, scale_factor=2, mode="bilinear", align_corners=True)
        if skip is not None:
            x = torch.cat([x, skip], dim=1)
        x = self.conv_block_1(x)
        x = self.conv_block_2(x)
        return x


In [ ]:
class UNetDecoder(nn.Module):
    '''
    Class that defines the full U-Net decoder over 5 encoder feature maps.

    Args:
        encoder_channels (List[int]): channels of [f1, f2, f3, f4, f5].
        decoder_channels (List[int]): output channels of the 5 decoder blocks.
    '''

    def __init__(self, encoder_channels: List[int],
                 decoder_channels: List[int] = [256, 128, 64, 32, 16]) -> None:
        super().__init__()
        c1, c2, c3, c4, c5 = encoder_channels
        d5, d4, d3, d2, d1 = decoder_channels

        self.decoder_block_5 = UNetDecoderBlock(m=c5, skip=c4, n=d5)
        self.decoder_block_4 = UNetDecoderBlock(m=d5, skip=c3, n=d4)
        self.decoder_block_3 = UNetDecoderBlock(m=d4, skip=c2, n=d3)
        self.decoder_block_2 = UNetDecoderBlock(m=d3, skip=c1, n=d2)
        self.decoder_block_1 = UNetDecoderBlock(m=d2, skip=0, n=d1)

    def forward(self, features: List[torch.Tensor]) -> torch.Tensor:
        f1, f2, f3, f4, f5 = features
        x = self.decoder_block_5(f5, f4)   # 1/16 resolution
        x = self.decoder_block_4(x, f3)    # 1/8 resolution
        x = self.decoder_block_3(x, f2)    # 1/4 resolution
        x = self.decoder_block_2(x, f1)    # 1/2 resolution
        x = self.decoder_block_1(x, None)  # full resolution
        return x


## 3. Final step, let's create the entire network

Now that we have the `ResNet18Encoder`, `UNetDecoder`, and the segmentation head,
it's time to assemble the full `UNetResNet18Model`!


In [ ]:
class UNetResNet18Model(nn.Module):
    '''
    Class that defines the full U-Net architecture with a ResNet18 encoder
    (~14.3M parameters).

    Args:
        Cin (int): number of input channels for the encoder.
        N (int): number of output channels (1 binary / num_classes semantic).
    '''

    def __init__(self, Cin: int = 3, N: int = 1) -> None:
        super().__init__()
        self.encoder = ResNet18Encoder(Cin=Cin)
        self.decoder = UNetDecoder(encoder_channels=self.encoder.out_channels,
                                   decoder_channels=[256, 128, 64, 32, 16])
        # 3x3 segmentation head that maps the last decoder features to N logits
        self.segmentation_head = nn.Conv2d(in_channels=16, out_channels=N,
                                           kernel_size=(3, 3), stride=(1, 1),
                                           padding=(1, 1))

    def forward(self, x) -> torch.Tensor:
        features = self.encoder(x)
        x = self.decoder(features)
        return self.segmentation_head(x)


### 3.1 Summary with images of shape `(B * 3 * 1024 * 1024)`

In [ ]:
from torchinfo import summary
# Full U-Net-ResNet18 model at 1024x1024
test_model = UNetResNet18Model(Cin=3, N=1)

summary(model = test_model,
        input_size=(1, 3, 1024, 1024), # (batch_size, num_channels, height, width)
        col_names = ["input_size", "output_size", "num_params", "trainable"],
        col_width = 20,
        row_settings = ["var_names"],
        depth = 3
        )


### 3.2 Summary with images of shape `(B * 3 * 512 * 512)`

In [ ]:
from torchinfo import summary
# Full U-Net-ResNet18 model at 512x512
test_model = UNetResNet18Model(Cin=3, N=1)

summary(model = test_model,
        input_size=(1, 3, 512, 512), # (batch_size, num_channels, height, width)
        col_names = ["input_size", "output_size", "num_params", "trainable"],
        col_width = 20,
        row_settings = ["var_names"],
        depth = 3
        )


## Integration with `src/models` and the training framework

The exact same classes above live in `src/models`, and the `unet-resnet18` model can be
built through the shared model factory. This is what the training scripts use when you pass
`--arch unet-resnet18`:

```bash
python scripts/train_binary_text.py --arch unet-resnet18
python scripts/train_semantic_layout.py --arch unet-resnet18
```

Let's double check the factory produces the same model, and run the IEEE efficiency
benchmark (parameters, FLOPs, inference latency/FPS, and peak memory - GPU if available,
otherwise CPU RSS) with `src.utils.benchmark`.


In [ ]:
import sys
from pathlib import Path
# Allow imports from the project root (src.*)
sys.path.insert(0, str(Path().cwd().parent))

from src.models import build_model

factory_model = build_model("unet-resnet18", Cin=3, N=1)
total_params = sum(p.numel() for p in factory_model.parameters())
print(f"U-Net-ResNet18 total parameters: {total_params:,} ({total_params/1e6:.2f}M)")


In [ ]:
from src.utils import benchmark_model, print_benchmark

device = "cuda" if torch.cuda.is_available() else "cpu"
report = benchmark_model(
    model=factory_model,
    input_size=(1, 3, 512, 512),
    device=device,
    warmup=5,
    iterations=20,
    arch_name="unet-resnet18",
)
print_benchmark(report)
